# Diagnóstico de calidad de datos de Instacart

**Actividad:** GA1-220501094-01-AA1-EV06 – Diagnosticar la calidad de datos  
**Rol:** analista de datos de Instacart  
**Aprendiz:** jeferson estiven vargas

> **Cómo está construido este notebook:** las cifras no están escritas a mano. Cada resultado se calcula con código a partir de los archivos que se cargan, así las conclusiones siempre corresponden a los datos reales.

## Momento 1. Reconocimiento del proyecto

**Contexto.** Instacart es una plataforma de compras de supermercado en línea. En la EV04 se exploraron sus datos para descubrir patrones de compra. Antes de sacar conclusiones más profundas hay que comprobar si esos datos son confiables.

**Datos disponibles.** El proyecto usa los cinco archivos CSV de Instacart guardados en la carpeta `data/` del repositorio (no se suben a GitHub). En conjunto describen los pedidos, los productos, los productos incluidos en cada pedido, los pasillos y los departamentos de la tienda.

**Objetivo del diagnóstico.** Evaluar la calidad de los datos antes de analizarlos, revisando:

1. La estructura de cada tabla (filas, columnas y tipos de datos).
2. Los valores faltantes.
3. Los tipos de datos incorrectos.
4. Los duplicados y las inconsistencias entre tablas.
5. Las limitaciones y las acciones de limpieza necesarias.

## Momento 2. Pregunta analítica y carga de datos

### Pregunta analítica

> ¿Tienen los cinco conjuntos de datos de Instacart la calidad suficiente (completitud, tipos correctos y consistencia entre tablas) para analizar los patrones de compra de los clientes, y qué acciones de limpieza se necesitan antes de hacerlo?

**Preguntas de apoyo:**

- ¿Qué columnas tienen valores faltantes y en qué proporción?
- ¿Hay columnas cuyo tipo de dato no corresponde a lo que representan?
- ¿Existen filas duplicadas o identificadores que no coinciden entre tablas?

In [1]:
from pathlib import Path
import csv

import pandas as pd

# El notebook puede ejecutarse desde la carpeta notebooks/ o desde la raíz del repositorio
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Si los CSV ya están en data/raw/ se usan desde ahí; si no, desde data/
CARPETA_RAW = RAIZ / "data" / "raw"
CARPETA_DATOS = CARPETA_RAW if any(CARPETA_RAW.glob("*.csv")) else RAIZ / "data"

print(f"Carpeta de datos: {CARPETA_DATOS}")

Carpeta de datos: c:\Users\estiv\OneDrive\Desktop\instacart-analisis\data


In [2]:
def detectar_separador(ruta: Path) -> str:
    """Lee la primera línea del archivo y detecta el separador de columnas."""
    with open(ruta, encoding="utf-8") as archivo:
        primera_linea = archivo.readline()
    try:
        return csv.Sniffer().sniff(primera_linea, delimiters=",;\t|").delimiter
    except csv.Error:
        return ","


def cargar_csv(ruta: Path) -> pd.DataFrame:
    """Carga un CSV usando el separador detectado."""
    return pd.read_csv(ruta, sep=detectar_separador(ruta))


rutas_csv = sorted(CARPETA_DATOS.glob("*.csv"))
if not rutas_csv:
    raise FileNotFoundError(
        f"No se encontraron archivos .csv en {CARPETA_DATOS}. "
        "Copie los datos de Instacart en la carpeta data/ antes de continuar."
    )

datasets = {}
for ruta in rutas_csv:
    try:
        datasets[ruta.stem] = cargar_csv(ruta)
    except (pd.errors.ParserError, UnicodeDecodeError) as error:
        print(f"No se pudo cargar {ruta.name}: {error}")

print(f"Archivos cargados: {list(datasets)}")

Archivos cargados: ['aisles', 'departments', 'instacart_orders', 'order_products', 'products']


In [3]:
resumen_carga = pd.DataFrame(
    [
        {
            "archivo": f"{nombre}.csv",
            "separador": repr(detectar_separador(CARPETA_DATOS / f"{nombre}.csv")),
            "filas": df.shape[0],
            "columnas": df.shape[1],
            "tamano_MB": round((CARPETA_DATOS / f"{nombre}.csv").stat().st_size / 1024**2, 2),
        }
        for nombre, df in datasets.items()
    ]
)
resumen_carga

,archivo,separador,filas,columnas,tamano_MB
0,aisles.csv,';',134,2,0.00
1,departments.csv,';',21,2,0.00
2,instacart_orders.csv,';',478967,6,11.74
3,order_products.csv,';',4545007,4,85.86
4,products.csv,';',49694,4,2.02


### Interpretación de la carga

In [4]:
total_filas = resumen_carga["filas"].sum()
mayor = resumen_carga.loc[resumen_carga["filas"].idxmax()]
menor = resumen_carga.loc[resumen_carga["filas"].idxmin()]

print(f"Se cargaron {len(datasets)} archivos con {total_filas:,} filas en total.")
print(f"La tabla más grande es {mayor['archivo']} ({mayor['filas']:,} filas) "
      f"y la más pequeña es {menor['archivo']} ({menor['filas']:,} filas).")
print("OK: se cargaron los 5 archivos esperados." if len(datasets) == 5
      else f"ATENCIÓN: se esperaban 5 archivos y se cargaron {len(datasets)}.")

Se cargaron 5 archivos con 5,073,823 filas en total.
La tabla más grande es order_products.csv (4,545,007 filas) y la más pequeña es departments.csv (21 filas).
OK: se cargaron los 5 archivos esperados.


## Momento 3. Inspección estructural de los datasets

En este momento se revisa la estructura de cada tabla con los métodos básicos de pandas:

| Método | Qué muestra |
|---|---|
| `head()` / `tail()` | Las primeras y las últimas filas, para ver cómo lucen los datos reales |
| `shape` | Cuántas filas y columnas tiene la tabla |
| `columns` | Los nombres de las columnas |
| `dtypes` | El tipo de dato de cada columna |
| `info()` | Resumen de tipos, valores no nulos y memoria usada |
| `describe()` | Estadísticas descriptivas (conteo, media, mínimo, máximo, etc.) |

In [5]:
from IPython.display import Markdown, display

for nombre, df in datasets.items():
    display(Markdown(f"### `{nombre}` · primeras y últimas 3 filas"))
    display(pd.concat([df.head(3), df.tail(3)]))

### `aisles` · primeras y últimas 3 filas

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
131,132,beauty
132,133,muscles joints pain relief
133,134,specialty wines champagnes


### `departments` · primeras y últimas 3 filas

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
18,19,snacks
19,20,deli
20,21,missing


### `instacart_orders` · primeras y últimas 3 filas

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
1,1690866,163593,5,5,12,9.0
2,1454967,39980,4,5,19,2.0
478964,885349,82944,16,2,11,6.0
478965,216274,4391,3,3,8,8.0
478966,2071924,1730,18,1,14,15.0


### `order_products` · primeras y últimas 3 filas

,order_id,product_id,add_to_cart_order,reordered
0,2141543,11440,17.0,0
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
4545004,692640,47766,4.0,1
4545005,319435,691,8.0,1
4545006,1398151,28733,9.0,0


### `products` · primeras y últimas 3 filas

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
49691,49692,ORGANIC INSTANT OATMEAL LIGHT MAPLE BROWN SUGAR,130,14
49692,49693,SPRING WATER BODY WASH,127,11
49693,49694,BURRITO- STEAK & CHEESE,38,1


### Dimensiones, columnas y tipos de datos

In [6]:
for nombre, df in datasets.items():
    print(f"{nombre}: shape = {df.shape}")
    print(f"  columnas: {list(df.columns)}")
    print(f"  dtypes:   {df.dtypes.astype(str).to_dict()}")
    print()

aisles: shape = (134, 2)
  columnas: ['aisle_id', 'aisle']
  dtypes:   {'aisle_id': 'int64', 'aisle': 'str'}

departments: shape = (21, 2)
  columnas: ['department_id', 'department']
  dtypes:   {'department_id': 'int64', 'department': 'str'}

instacart_orders: shape = (478967, 6)
  columnas: ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
  dtypes:   {'order_id': 'int64', 'user_id': 'int64', 'order_number': 'int64', 'order_dow': 'int64', 'order_hour_of_day': 'int64', 'days_since_prior_order': 'float64'}

order_products: shape = (4545007, 4)
  columnas: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']
  dtypes:   {'order_id': 'int64', 'product_id': 'int64', 'add_to_cart_order': 'float64', 'reordered': 'int64'}

products: shape = (49694, 4)
  columnas: ['product_id', 'product_name', 'aisle_id', 'department_id']
  dtypes:   {'product_id': 'int64', 'product_name': 'str', 'aisle_id': 'int64', 'department_id': 'int64'}



In [7]:
tipos_por_columna = pd.DataFrame(
    [
        {"dataset": nombre, "columna": columna, "tipo": str(tipo)}
        for nombre, df in datasets.items()
        for columna, tipo in df.dtypes.items()
    ]
)
tipos_por_columna

,dataset,columna,tipo
0,aisles,aisle_id,int64
1,aisles,aisle,str
2,departments,department_id,int64
3,departments,department,str
4,instacart_orders,order_id,int64
5,instacart_orders,user_id,int64
6,instacart_orders,order_number,int64
7,instacart_orders,order_dow,int64
8,instacart_orders,order_hour_of_day,int64
9,instacart_orders,days_since_prior_order,float64


### Resumen con `info()`

In [8]:
for nombre, df in datasets.items():
    display(Markdown(f"#### `{nombre}`"))
    df.info(memory_usage="deep")

#### `aisles`

<class 'pandas.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   aisle_id  134 non-null    int64
 1   aisle     134 non-null    str  
dtypes: int64(1), str(1)
memory usage: 9.6 KB


#### `departments`

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   department_id  21 non-null     int64
 1   department     21 non-null     str  
dtypes: int64(1), str(1)
memory usage: 1.5 KB


#### `instacart_orders`

<class 'pandas.DataFrame'>
RangeIndex: 478967 entries, 0 to 478966
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                478967 non-null  int64  
 1   user_id                 478967 non-null  int64  
 2   order_number            478967 non-null  int64  
 3   order_dow               478967 non-null  int64  
 4   order_hour_of_day       478967 non-null  int64  
 5   days_since_prior_order  450148 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 21.9 MB


#### `order_products`

<class 'pandas.DataFrame'>
RangeIndex: 4545007 entries, 0 to 4545006
Data columns (total 4 columns):
 #   Column             Dtype  
---  ------             -----  
 0   order_id           int64  
 1   product_id         int64  
 2   add_to_cart_order  float64
 3   reordered          int64  
dtypes: float64(1), int64(3)
memory usage: 138.7 MB


#### `products`

<class 'pandas.DataFrame'>
RangeIndex: 49694 entries, 0 to 49693
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   product_id     49694 non-null  int64
 1   product_name   48436 non-null  str  
 2   aisle_id       49694 non-null  int64
 3   department_id  49694 non-null  int64
dtypes: int64(3), str(1)
memory usage: 4.9 MB


### Estadísticas descriptivas con `describe()`

In [9]:
for nombre, df in datasets.items():
    display(Markdown(f"#### `{nombre}`"))
    if not df.select_dtypes(include="number").empty:
        display(df.describe().T)
    if not df.select_dtypes(include="object").empty:
        display(df.describe(include="object").T)

#### `aisles`

,count,mean,std,min,25%,50%,75%,max
aisle_id,134.0,67.5,38.826537,1.0,34.25,67.5,100.75,134.0


C:\Users\estiv\AppData\Local\Temp\ipykernel_26484\2644845588.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  if not df.select_dtypes(include="object").empty:
C:\Users\estiv\AppData\Local\Temp\ipykernel_26484\2644845588.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migratio

,count,unique,top,freq
aisle,134,134,prepared soups salads,1


#### `departments`

,count,mean,std,min,25%,50%,75%,max
department_id,21.0,11.0,6.204837,1.0,6.0,11.0,16.0,21.0


... (6 outputs left out)

#### `products`

,count,mean,std,min,25%,50%,75%,max
product_id,49694.0,24847.500000,14345.566475,1.0,12424.25,24847.5,37270.75,49694.0
aisle_id,49694.0,67.773192,38.317265,1.0,35.00,69.0,100.00,134.0
department_id,49694.0,11.728478,5.850314,1.0,7.00,13.0,17.00,21.0


C:\Users\estiv\AppData\Local\Temp\ipykernel_26484\2644845588.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  if not df.select_dtypes(include="object").empty:
C:\Users\estiv\AppData\Local\Temp\ipykernel_26484\2644845588.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migratio

,count,unique,top,freq
product_name,48436,48436,Chocolate Sandwich Cookies,1


### Tabla del diagnóstico estructural

In [10]:
diagnostico_estructural = pd.DataFrame(
    [
        {
            "dataset": nombre,
            "filas": df.shape[0],
            "columnas": df.shape[1],
            "columnas_numericas": df.select_dtypes(include="number").shape[1],
            "columnas_texto": df.select_dtypes(include="object").shape[1],
            "memoria_MB": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
        }
        for nombre, df in datasets.items()
    ]
)
diagnostico_estructural

C:\Users\estiv\AppData\Local\Temp\ipykernel_26484\1649051297.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  "columnas_texto": df.select_dtypes(include="object").shape[1],
C:\Users\estiv\AppData\Local\Temp\ipykernel_26484\1649051297.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_

,dataset,filas,columnas,columnas_numericas,columnas_texto,memoria_MB
0,aisles,134,2,1,1,0.01
1,departments,21,2,1,1,0.00
2,instacart_orders,478967,6,6,0,21.93
3,order_products,4545007,4,4,0,138.70
4,products,49694,4,3,1,4.87


### Interpretación de la inspección

In [11]:
mas_memoria = diagnostico_estructural.loc[diagnostico_estructural["memoria_MB"].idxmax()]
print(f"La tabla que más memoria usa es '{mas_memoria['dataset']}' "
      f"con {mas_memoria['memoria_MB']} MB.")

# Columnas que deberían ser enteras (identificadores y contadores) pero quedaron como decimales
columnas_float = tipos_por_columna[tipos_por_columna["tipo"].str.startswith("float")]
if columnas_float.empty:
    print("Ninguna columna quedó como decimal (float).")
else:
    print("Columnas cargadas como decimales (float); suele indicar valores faltantes, se revisa en el Momento 4:")
    for _, fila in columnas_float.iterrows():
        print(f"  - {fila['dataset']}.{fila['columna']}")

columnas_texto = tipos_por_columna[tipos_por_columna["tipo"].isin(["object", "str", "string"])]
print(f"Hay {len(columnas_texto)} columnas de texto: "
      f"{[f'{d}.{c}' for d, c in zip(columnas_texto['dataset'], columnas_texto['columna'])]}")

La tabla que más memoria usa es 'order_products' con 138.7 MB.
Columnas cargadas como decimales (float); suele indicar valores faltantes, se revisa en el Momento 4:
  - instacart_orders.days_since_prior_order
  - order_products.add_to_cart_order
Hay 3 columnas de texto: ['aisles.aisle', 'departments.department', 'products.product_name']


## Momento 4. Diagnóstico de calidad de los datos

El diagnóstico se divide en tres revisiones:

- **4.1 Valores faltantes:** celdas vacías (`NaN`) por columna.
- **4.2 Tipos de datos:** columnas cuyo tipo no corresponde a lo que representan.
- **4.3 Inconsistencias:** duplicados, identificadores que no existen en la tabla relacionada, valores fuera de rango y problemas en los textos.

### 4.1 Valores faltantes

In [ ]:
faltantes = pd.DataFrame(
    [
        {
            "dataset": nombre,
            "columna": columna,
            "faltantes": int(df[columna].isna().sum()),
            "porcentaje": round(df[columna].isna().mean() * 100, 2),
        }
        for nombre, df in datasets.items()
        for columna in df.columns
    ]
)
columnas_con_faltantes = faltantes[faltantes["faltantes"] > 0].sort_values("porcentaje", ascending=False)
columnas_con_faltantes

In [ ]:
if columnas_con_faltantes.empty:
    print("Ninguna columna tiene valores faltantes.")
else:
    print(f"{len(columnas_con_faltantes)} columnas tienen valores faltantes:")
    for _, fila in columnas_con_faltantes.iterrows():
        print(f"  - {fila['dataset']}.{fila['columna']}: {fila['faltantes']:,} faltantes ({fila['porcentaje']}%)")

### 4.2 Tipos de datos

Regla usada: los identificadores (`*_id`) y los contadores (número de pedido, posición en el carrito, día, hora, recompra) deberían ser **enteros**. Si aparecen como decimales (`float`), casi siempre es porque la columna tiene faltantes: pandas no puede guardar `NaN` en una columna `int64`.

In [ ]:
COLUMNAS_ENTERAS = {
    "order_number", "add_to_cart_order", "order_dow",
    "order_hour_of_day", "days_since_prior_order", "reordered",
}


def tipo_esperado(columna: str) -> str:
    """Devuelve el tipo que debería tener una columna según lo que representa."""
    if columna.endswith("_id") or columna in COLUMNAS_ENTERAS:
        return "entero"
    return "texto"


def tipo_actual(serie: pd.Series) -> str:
    if pd.api.types.is_integer_dtype(serie):
        return "entero"
    if pd.api.types.is_float_dtype(serie):
        return "decimal"
    return "texto"


revision_tipos = pd.DataFrame(
    [
        {
            "dataset": nombre,
            "columna": columna,
            "tipo_pandas": str(df[columna].dtype),
            "tipo_actual": tipo_actual(df[columna]),
            "tipo_esperado": tipo_esperado(columna),
        }
        for nombre, df in datasets.items()
        for columna in df.columns
    ]
)
tipos_incorrectos = revision_tipos[revision_tipos["tipo_actual"] != revision_tipos["tipo_esperado"]]
tipos_incorrectos

In [ ]:
if tipos_incorrectos.empty:
    print("Todas las columnas tienen el tipo esperado.")
else:
    print(f"{len(tipos_incorrectos)} columnas no tienen el tipo esperado:")
    for _, fila in tipos_incorrectos.iterrows():
        con_faltantes = faltantes.query("dataset == @fila.dataset and columna == @fila.columna")["faltantes"].iloc[0]
        causa = f"tiene {con_faltantes:,} faltantes" if con_faltantes else "no tiene faltantes; revisar valores"
        print(f"  - {fila['dataset']}.{fila['columna']}: es {fila['tipo_actual']} ({fila['tipo_pandas']}), "
              f"debería ser {fila['tipo_esperado']} ({causa})")

### 4.3 Inconsistencias

#### Duplicados

Se revisan dos tipos de duplicados: filas idénticas completas y claves repetidas (un identificador que debería ser único y aparece más de una vez).

In [ ]:
def buscar_dataset(palabra: str, excluir) -> str | None:
    """Devuelve el primer dataset cuyo nombre contiene la palabra y que no esté en excluir."""
    return next((nombre for nombre in datasets if palabra in nombre and nombre not in excluir), None)


# Clave que debería ser única en cada tabla de Instacart
CLAVES = {
    "order_products": ["order_id", "product_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "aisles": ["aisle_id"],
    "departments": ["department_id"],
}

nombres_claves = {}   # nombre real del archivo -> columnas clave
dataset_de = {}       # palabra (orders, products...) -> nombre real del archivo
for palabra, clave in CLAVES.items():
    nombre = buscar_dataset(palabra, excluir=nombres_claves)
    if nombre and set(clave) <= set(datasets[nombre].columns):
        nombres_claves[nombre] = clave
        dataset_de[palabra] = nombre

duplicados = pd.DataFrame(
    [
        {
            "dataset": nombre,
            "filas_duplicadas": int(df.duplicated().sum()),
            "clave": ", ".join(nombres_claves.get(nombre, [])) or "sin clave definida",
            "claves_repetidas": int(df.duplicated(subset=nombres_claves[nombre]).sum()) if nombre in nombres_claves else 0,
        }
        for nombre, df in datasets.items()
    ]
)
duplicados

#### Integridad entre tablas

Cada identificador de una tabla hija debe existir en su tabla principal. Por ejemplo, todo `product_id` de los pedidos debe existir en la tabla de productos.

In [ ]:
RELACIONES = [
    ("order_products", "order_id", "orders"),
    ("order_products", "product_id", "products"),
    ("products", "aisle_id", "aisles"),
    ("products", "department_id", "departments"),
]

filas_integridad = []
for palabra_hija, columna, palabra_padre in RELACIONES:
    hija = dataset_de.get(palabra_hija)
    padre = dataset_de.get(palabra_padre)
    if not hija or not padre or columna not in datasets[hija].columns:
        continue
    valores_hija = datasets[hija][columna].dropna()
    huerfanos = ~valores_hija.isin(datasets[padre][columna])
    filas_integridad.append({
        "relacion": f"{hija}.{columna} → {padre}.{columna}",
        "filas_revisadas": len(valores_hija),
        "ids_sin_coincidencia": int(huerfanos.sum()),
    })

integridad = pd.DataFrame(filas_integridad)
integridad

#### Valores fuera de rango

In [ ]:
RANGOS_VALIDOS = {
    "order_dow": (0, 6),
    "order_hour_of_day": (0, 23),
    "days_since_prior_order": (0, 30),
    "reordered": (0, 1),
    "add_to_cart_order": (1, None),
    "order_number": (1, None),
}

filas_rangos = []
for nombre, df in datasets.items():
    for columna, (minimo, maximo) in RANGOS_VALIDOS.items():
        if columna not in df.columns:
            continue
        valores = df[columna].dropna()
        fuera = (valores < minimo) if maximo is None else ~valores.between(minimo, maximo)
        filas_rangos.append({
            "dataset": nombre,
            "columna": columna,
            "rango_valido": f"{minimo} a {maximo if maximo is not None else 'sin límite'}",
            "minimo_real": valores.min(),
            "maximo_real": valores.max(),
            "fuera_de_rango": int(fuera.sum()),
        })

rangos = pd.DataFrame(filas_rangos)
rangos

#### Problemas en columnas de texto

Se revisan espacios al inicio o al final, nombres que solo cambian en mayúsculas/minúsculas y valores de relleno como `missing` o `unknown`.

In [ ]:
VALORES_RELLENO = {"missing", "unknown", "none", "nan", "n/a", ""}

filas_texto = []
for nombre, df in datasets.items():
    for columna in df.columns:
        if tipo_actual(df[columna]) != "texto":
            continue
        valores = df[columna].dropna().astype(str)
        filas_texto.append({
            "dataset": nombre,
            "columna": columna,
            "con_espacios_extra": int((valores != valores.str.strip()).sum()),
            "duplicados_por_mayusculas": int(valores.nunique() - valores.str.strip().str.lower().nunique()),
            "valores_de_relleno": int(valores.str.strip().str.lower().isin(VALORES_RELLENO).sum()),
        })

problemas_texto = pd.DataFrame(filas_texto)
problemas_texto

### Interpretación del Momento 4

In [ ]:
print("4.1 Faltantes:")
print(f"  {len(columnas_con_faltantes)} columnas con faltantes, "
      f"{int(columnas_con_faltantes['faltantes'].sum()):,} celdas vacías en total.")

print("4.2 Tipos:")
print(f"  {len(tipos_incorrectos)} columnas con un tipo distinto al esperado.")

print("4.3 Inconsistencias:")
for _, fila in duplicados.iterrows():
    if fila["filas_duplicadas"] or fila["claves_repetidas"]:
        print(f"  - {fila['dataset']}: {fila['filas_duplicadas']:,} filas duplicadas y "
              f"{fila['claves_repetidas']:,} claves repetidas ({fila['clave']}).")
for _, fila in integridad[integridad["ids_sin_coincidencia"] > 0].iterrows():
    print(f"  - {fila['relacion']}: {fila['ids_sin_coincidencia']:,} ids sin coincidencia.")
for _, fila in rangos[rangos["fuera_de_rango"] > 0].iterrows():
    print(f"  - {fila['dataset']}.{fila['columna']}: {fila['fuera_de_rango']:,} valores fuera de {fila['rango_valido']}.")
for _, fila in problemas_texto.iterrows():
    total = fila["con_espacios_extra"] + fila["duplicados_por_mayusculas"] + fila["valores_de_relleno"]
    if total:
        print(f"  - {fila['dataset']}.{fila['columna']}: {fila['con_espacios_extra']} con espacios extra, "
              f"{fila['duplicados_por_mayusculas']} duplicados por mayúsculas, "
              f"{fila['valores_de_relleno']} valores de relleno.")

### 4.4 Tabla de hallazgos y acciones propuestas

Cada problema detectado en 4.1 a 4.3 se convierte en un hallazgo con:

- **dimensión de calidad:** completitud, validez, unicidad, integridad o consistencia;
- **impacto:** según el porcentaje de filas afectadas (**alto** ≥ 5 %, **medio** ≥ 1 %, **bajo** < 1 %);
- **acción propuesta:** qué haría un analista antes de usar los datos.

> En esta actividad solo se **diagnostica**: las acciones se proponen, pero no se aplican a los datos.

In [ ]:
def nivel_impacto(porcentaje: float) -> str:
    if porcentaje >= 5:
        return "Alto"
    if porcentaje >= 1:
        return "Medio"
    return "Bajo"


def porcentaje_de(cantidad: int, dataset: str) -> float:
    return round(cantidad / len(datasets[dataset]) * 100, 2)


def accion_para_faltantes(dataset: str, columna: str) -> str:
    if columna == "days_since_prior_order":
        return ("No eliminar: el faltante corresponde al primer pedido de cada cliente. "
                "Convertir a Int64 o crear una columna es_primer_pedido.")
    if tipo_esperado(columna) == "texto":
        return "Reemplazar por la categoría 'Unknown' para no perder las filas."
    return "Revisar la causa; excluir esas filas en los cálculos que usen la columna o imputar con criterio justificado."


filas_hallazgos = []


def agregar(dimension, dataset, columna, descripcion, cantidad, accion):
    porcentaje = porcentaje_de(cantidad, dataset)
    filas_hallazgos.append({
        "dimension": dimension,
        "dataset": dataset,
        "columna": columna,
        "descripcion": descripcion,
        "filas_afectadas": cantidad,
        "porcentaje": porcentaje,
        "impacto": nivel_impacto(porcentaje),
        "accion_propuesta": accion,
    })


for _, f in columnas_con_faltantes.iterrows():
    agregar("Completitud", f["dataset"], f["columna"], "Valores faltantes (NaN)",
            f["faltantes"], accion_para_faltantes(f["dataset"], f["columna"]))

for _, f in tipos_incorrectos.iterrows():
    cantidad = int(datasets[f["dataset"]][f["columna"]].isna().sum())
    agregar("Validez (tipo)", f["dataset"], f["columna"],
            f"Es {f['tipo_actual']} ({f['tipo_pandas']}) y debería ser {f['tipo_esperado']}",
            cantidad, "Convertir con astype('Int64'), que admite faltantes, después de tratar los NaN.")

for _, f in duplicados.iterrows():
    if f["filas_duplicadas"]:
        agregar("Unicidad", f["dataset"], "(fila completa)", "Filas idénticas repetidas",
                f["filas_duplicadas"], "Eliminar con drop_duplicates() conservando la primera aparición.")
    repetidas_no_identicas = f["claves_repetidas"] - f["filas_duplicadas"]
    if repetidas_no_identicas > 0:
        agregar("Unicidad", f["dataset"], f["clave"], "Clave repetida con datos distintos",
                repetidas_no_identicas, "Revisar cada caso antes de eliminar; decidir qué registro es el válido.")

for _, f in integridad[integridad["ids_sin_coincidencia"] > 0].iterrows():
    hija = f["relacion"].split(".")[0]
    columna = f["relacion"].split(".")[1].split(" ")[0]
    agregar("Integridad", hija, columna, f"Ids sin coincidencia: {f['relacion']}",
            f["ids_sin_coincidencia"], "Excluir esas filas de los cruces (merge) o completar la tabla principal.")

for _, f in rangos[rangos["fuera_de_rango"] > 0].iterrows():
    agregar("Validez (rango)", f["dataset"], f["columna"], f"Valores fuera del rango {f['rango_valido']}",
            f["fuera_de_rango"], "Revisar el origen; corregir si es un error de captura o excluir del análisis.")

for _, f in problemas_texto.iterrows():
    if f["con_espacios_extra"]:
        agregar("Consistencia (texto)", f["dataset"], f["columna"], "Espacios al inicio o al final",
                f["con_espacios_extra"], "Limpiar con str.strip().")
    if f["duplicados_por_mayusculas"]:
        agregar("Consistencia (texto)", f["dataset"], f["columna"], "Valores que solo cambian en mayúsculas",
                f["duplicados_por_mayusculas"], "Normalizar con str.strip().str.lower() antes de agrupar.")
    if f["valores_de_relleno"]:
        agregar("Consistencia (texto)", f["dataset"], f["columna"], "Valores de relleno (missing, unknown…)",
                f["valores_de_relleno"], "Tratarlos como faltantes o como una categoría explícita 'Unknown'.")

COLUMNAS_HALLAZGOS = ["dimension", "dataset", "columna", "descripcion",
                      "filas_afectadas", "porcentaje", "impacto", "accion_propuesta"]
orden_impacto = {"Alto": 0, "Medio": 1, "Bajo": 2}
hallazgos_calidad = (
    pd.DataFrame(filas_hallazgos, columns=COLUMNAS_HALLAZGOS)
    .sort_values(["impacto", "porcentaje"], key=lambda s: s.map(orden_impacto) if s.name == "impacto" else -s)
    .reset_index(drop=True)
)
hallazgos_calidad.index = hallazgos_calidad.index + 1
hallazgos_calidad.index.name = "hallazgo"
hallazgos_calidad

In [ ]:
if hallazgos_calidad.empty:
    print("No se encontraron problemas de calidad con las reglas aplicadas.")
else:
    print(f"Se registraron {len(hallazgos_calidad)} hallazgos de calidad.")
    print("Por impacto:   ", hallazgos_calidad["impacto"].value_counts().reindex(orden_impacto, fill_value=0).to_dict())
    print("Por dimensión: ", hallazgos_calidad["dimension"].value_counts().to_dict())
    principal = hallazgos_calidad.iloc[0]
    print(f"Hallazgo de mayor impacto: {principal['dataset']}.{principal['columna']} — "
          f"{principal['descripcion']} ({principal['porcentaje']}% de las filas).")

### 4.5 Limitaciones del diagnóstico y de los datos

**Limitaciones de los datos:**

- No hay fechas reales de los pedidos: solo el día de la semana (`order_dow`), la hora (`order_hour_of_day`) y los días desde el pedido anterior. No se pueden estudiar tendencias por mes o año ni temporadas.
- No hay precios, cantidades compradas ni información de los clientes (edad, ubicación). La pregunta analítica se limita a patrones de frecuencia y de productos.
- `days_since_prior_order` parece estar limitado a un máximo de 30 días, así que los clientes que tardan más en volver quedan agrupados en ese valor.

**Limitaciones del diagnóstico:**

- Los rangos válidos (0–6 para el día, 0–23 para la hora, 0–30 días, 0/1 para la recompra) son reglas asumidas a partir del significado de cada columna; no vienen de un diccionario de datos oficial.
- La revisión de textos detecta espacios, mayúsculas y valores de relleno, pero no errores de ortografía ni productos con nombres parecidos.
- Se diagnostica, pero no se limpia: las cifras del análisis posterior pueden cambiar cuando se apliquen las acciones propuestas.